# Section 04: Preparing and Using Data for Analysis

**CareerAlign GCP Professional Data Engineer**

This notebook covers:
1. BI Engine and materialized views for dashboards
2. BigQuery ML end-to-end workflow (CREATE MODEL, EVALUATE, PREDICT)
3. BQML TRANSFORM clause for feature engineering
4. Embeddings and vector search patterns
5. Analytics Hub data sharing
6. Cloud DLP and dynamic data masking

> **Note**: Cells marked with `# REQUIRES GCP` need a GCP project. Local-only cells run anywhere.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rajvermatx/careeralign.com/blob/main/gcp-pde/notebooks/04-preparing-using-data.ipynb)

---
## Setup & Installations

In [ ]:
!pip install -q google-cloud-bigquery google-cloud-dlp pandas numpy scikit-learn

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from sklearn.preprocessing import LabelEncoder, StandardScaler
import warnings
warnings.filterwarnings('ignore')

print("Setup complete!")

---
## 1. BI Engine and Materialized Views

Show how materialized views and BI Engine accelerate dashboards.

In [ ]:
# --- Materialized View SQL Examples ---

mv_sql = """
-- Create materialized view for dashboard aggregations
CREATE MATERIALIZED VIEW `project.dataset.mv_daily_metrics`
PARTITION BY event_date
CLUSTER BY region
OPTIONS (
  enable_refresh = true,
  refresh_interval_minutes = 30,
  max_staleness = INTERVAL "1:0:0" HOUR TO SECOND
)
AS
SELECT
  DATE(event_time) AS event_date,
  region,
  product_category,
  COUNT(*) AS total_events,
  COUNT(DISTINCT user_id) AS unique_users,
  SUM(revenue) AS total_revenue,
  AVG(session_duration) AS avg_session_sec,
  APPROX_COUNT_DISTINCT(session_id) AS approx_sessions
FROM `project.dataset.events`
GROUP BY 1, 2, 3;

-- This query auto-rewrites to use the MV:
SELECT region, SUM(total_revenue) AS revenue
FROM `project.dataset.events`
WHERE DATE(event_time) BETWEEN '2025-02-01' AND '2025-02-28'
GROUP BY region;

-- BI Engine reservation
-- bq update --project_id=my-project --bi_reservation_size=10GB --location=US
"""

print("=== Materialized Views + BI Engine ===")
print(mv_sql)
print("Key points:")
print("  - max_staleness: serve slightly stale data for zero compute cost")
print("  - Auto-rewrite: BQ rewrites queries to use MVs transparently")
print("  - BI Engine: in-memory cache, sub-second responses")
print("  - Combined: MV pre-aggregates -> BI Engine caches -> instant dashboards")

---
## 2. BigQuery ML Workflow

Simulate the BQML workflow locally with scikit-learn, showing equivalent SQL.

In [ ]:
# --- Create training dataset ---
np.random.seed(42)
n = 2000

df_ml = pd.DataFrame({
    'user_tenure_days': np.random.randint(1, 1000, n),
    'total_purchases': np.random.poisson(10, n),
    'avg_session_min': np.random.lognormal(2, 0.5, n).round(1),
    'support_tickets': np.random.poisson(2, n),
    'last_login_days': np.random.exponential(30, n).astype(int),
    'subscription': np.random.choice(['free', 'basic', 'premium'], n, p=[0.5, 0.35, 0.15]),
})

# Create label (churn) based on features
churn_prob = (
    0.3 * (df_ml['last_login_days'] > 60).astype(float) +
    0.2 * (df_ml['support_tickets'] > 3).astype(float) +
    0.2 * (df_ml['total_purchases'] < 5).astype(float) +
    0.15 * (df_ml['subscription'] == 'free').astype(float) +
    np.random.uniform(0, 0.3, n)
)
df_ml['churned'] = (churn_prob > 0.5).astype(int)

print(f"Training dataset: {n} rows")
print(f"Churn rate: {df_ml['churned'].mean():.1%}")
print(f"Features: {[c for c in df_ml.columns if c != 'churned']}")
df_ml.head()

In [ ]:
# --- Train model (equivalent to BQML CREATE MODEL) ---

le = LabelEncoder()
df_ml['subscription_encoded'] = le.fit_transform(df_ml['subscription'])

features = ['user_tenure_days', 'total_purchases', 'avg_session_min',
            'support_tickets', 'last_login_days', 'subscription_encoded']

X = df_ml[features].values
y = df_ml['churned'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

model = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
model.fit(X_train_s, y_train)

y_pred = model.predict(X_test_s)
y_proba = model.predict_proba(X_test_s)[:, 1]

print("=== Model Training (equiv: CREATE MODEL ... OPTIONS(model_type='LOGISTIC_REG')) ===")
print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print()

# Equivalent BQML SQL
bqml_create = """
-- BQML equivalent:
CREATE OR REPLACE MODEL `project.dataset.churn_model`
OPTIONS (
  model_type = 'LOGISTIC_REG',
  input_label_cols = ['churned'],
  auto_class_weights = TRUE,
  data_split_method = 'AUTO_SPLIT',
  enable_global_explain = TRUE
) AS
SELECT * FROM `project.dataset.user_features`;
"""
print(bqml_create)

In [ ]:
# --- Evaluate model (equivalent to ML.EVALUATE) ---

print("=== Model Evaluation (equiv: SELECT * FROM ML.EVALUATE(MODEL ...)) ===")
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC AUC:   {roc_auc_score(y_test, y_proba):.4f}")
print()
print(classification_report(y_test, y_pred, target_names=['Not Churned', 'Churned']))

# Feature importance (equivalent to ML.GLOBAL_EXPLAIN)
print("=== Feature Importance (equiv: ML.GLOBAL_EXPLAIN) ===")
importance = pd.DataFrame({
    'feature': features,
    'coefficient': model.coef_[0],
    'abs_importance': np.abs(model.coef_[0])
}).sort_values('abs_importance', ascending=False)

for _, row in importance.iterrows():
    direction = '+' if row['coefficient'] > 0 else '-'
    bar = '#' * int(row['abs_importance'] * 20)
    print(f"  {row['feature']:<25} {direction}{row['abs_importance']:.3f} {bar}")

In [ ]:
# --- Predict (equivalent to ML.PREDICT) ---

predictions = pd.DataFrame({
    'user_idx': range(len(y_test)),
    'actual_churned': y_test,
    'predicted_churned': y_pred,
    'churn_probability': y_proba.round(4)
})

print("=== Predictions (equiv: SELECT * FROM ML.PREDICT(MODEL ...)) ===")
print(predictions.head(15).to_string(index=False))
print()
print("-- BQML equivalent:")
print("SELECT user_id, predicted_churned, predicted_churned_probs")
print("FROM ML.PREDICT(MODEL `project.dataset.churn_model`,")
print("  (SELECT * FROM `project.dataset.new_users`));")

---
## 3. BQML TRANSFORM Clause

Show how TRANSFORM embeds feature engineering into the model.

In [ ]:
# --- BQML TRANSFORM examples ---

transform_sql = """
-- TRANSFORM: Feature engineering embedded in the model
-- Applied automatically at ML.PREDICT time!

CREATE OR REPLACE MODEL `project.dataset.churn_with_transform`
TRANSFORM(
  -- Scaling
  ML.STANDARD_SCALER(user_tenure_days) OVER() AS scaled_tenure,
  ML.STANDARD_SCALER(avg_session_min) OVER() AS scaled_session,
  
  -- Bucketizing
  ML.BUCKETIZE(total_purchases, [0, 5, 10, 25, 50]) AS purchase_bucket,
  ML.BUCKETIZE(last_login_days, [0, 7, 30, 60, 90]) AS login_recency,
  
  -- Feature crosses
  ML.FEATURE_CROSS(STRUCT(subscription, login_recency)) AS sub_x_login,
  
  -- Quantile bucketizing
  ML.QUANTILE_BUCKETIZE(support_tickets, 4) OVER() AS ticket_quartile,
  
  -- Missing value handling
  IFNULL(last_purchase_amount, 0) AS last_purchase_imputed,
  IF(last_purchase_amount IS NULL, 1, 0) AS purchase_missing_flag,
  
  -- Label
  churned
)
OPTIONS (
  model_type = 'BOOSTED_TREE_CLASSIFIER',
  input_label_cols = ['churned'],
  auto_class_weights = TRUE,
  num_parallel_tree = 10,
  max_iterations = 50
) AS
SELECT * FROM `project.dataset.user_features`;


-- BQML Model Types Quick Reference:
--   LINEAR_REG          - Linear regression
--   LOGISTIC_REG        - Logistic regression
--   KMEANS              - K-means clustering
--   ARIMA_PLUS          - Time series forecasting
--   BOOSTED_TREE_*      - XGBoost (classifier/regressor)
--   DNN_*               - Deep neural network
--   AUTOML_*            - AutoML Tables
--   MATRIX_FACTORIZATION - Recommendations
--   TENSORFLOW          - Import pre-trained TF model
--   REMOTE              - Call Vertex AI endpoint
"""

print("=== BQML TRANSFORM Clause ===")
print(transform_sql)

---
## 4. Embeddings and Vector Search

Show BigQuery SQL patterns for embeddings, vector search, and RAG.

In [ ]:
# --- Embeddings + Vector Search SQL ---

embedding_sql = """
-- Step 1: Create remote model for embeddings
CREATE OR REPLACE MODEL `project.dataset.embedding_model`
REMOTE WITH CONNECTION `us.vertex-ai-connection`
OPTIONS (endpoint = 'text-embedding-005');

-- Step 2: Generate embeddings for documents
CREATE TABLE `project.dataset.doc_embeddings` AS
SELECT
  doc_id, title, content,
  ml_generate_embedding_result AS embedding
FROM ML.GENERATE_EMBEDDING(
  MODEL `project.dataset.embedding_model`,
  (SELECT doc_id, title, content FROM `project.dataset.documents`),
  STRUCT(TRUE AS flatten_json_output)
);

-- Step 3: Vector search (find similar documents)
SELECT base.doc_id, base.title, distance
FROM VECTOR_SEARCH(
  TABLE `project.dataset.doc_embeddings`, 'embedding',
  (SELECT ml_generate_embedding_result AS embedding
   FROM ML.GENERATE_EMBEDDING(
     MODEL `project.dataset.embedding_model`,
     (SELECT 'How to optimize BigQuery costs' AS content)
   )),
  top_k => 5,
  distance_type => 'COSINE'
);
"""

print("=== BigQuery Embeddings + Vector Search ===")
print(embedding_sql)

In [ ]:
# --- Simulate vector search locally ---

# Generate fake embeddings (in reality these come from a model)
np.random.seed(42)
docs = [
    'BigQuery partitioning and clustering best practices',
    'How to optimize Dataflow streaming pipelines',
    'Cloud Spanner global transactions explained',
    'Bigtable row key design patterns',
    'Cloud Composer DAG scheduling strategies',
    'BigQuery cost reduction techniques',
    'Pub/Sub message ordering and dead-letter topics',
    'Data lake governance with Dataplex',
]

# Simple bag-of-words "embedding" for demonstration
from collections import Counter
vocab = set()
for doc in docs:
    vocab.update(doc.lower().split())
vocab = sorted(vocab)

def simple_embedding(text):
    words = Counter(text.lower().split())
    vec = np.array([words.get(w, 0) for w in vocab], dtype=float)
    norm = np.linalg.norm(vec)
    return vec / norm if norm > 0 else vec

embeddings = np.array([simple_embedding(doc) for doc in docs])

# Query
query = 'BigQuery cost optimization'
query_emb = simple_embedding(query)

# Cosine similarity
similarities = embeddings @ query_emb
results = sorted(zip(docs, similarities), key=lambda x: -x[1])

print(f"Query: '{query}'")
print(f"\nTop-5 similar documents (cosine similarity):")
for i, (doc, sim) in enumerate(results[:5], 1):
    print(f"  {i}. [{sim:.3f}] {doc}")

---
## 5. Analytics Hub Data Sharing

In [ ]:
# --- Analytics Hub Concepts ---

analytics_hub = """
=== Analytics Hub: Zero-Copy Data Sharing ===

Concept Hierarchy:
  Data Exchange (container for listings)
    -> Listing (published dataset + metadata)
      -> Linked Dataset (subscriber's read-only view)

How it works:
  1. Publisher creates a Data Exchange
  2. Publisher adds a Listing (points to a BigQuery dataset)
  3. Subscriber discovers and subscribes to the listing
  4. Subscriber gets a Linked Dataset (zero-copy, read-only)
  5. Subscriber queries the linked dataset (pays for their own queries)

Cost model:
  - Publisher pays for: storage
  - Subscriber pays for: compute (queries)
  - No data copying or ETL needed!

Security:
  - Publisher controls who can subscribe
  - Row-level and column-level security enforced
  - Audit logs for all access

vs Authorized Views:
  - Authorized views: share within same org, view-level access
  - Analytics Hub: share across orgs, dataset-level, marketplace

# gcloud commands:
gcloud bigquery analytics-hub exchanges create my-exchange \\
  --location=US --display-name="My Data Exchange"

gcloud bigquery analytics-hub listings create my-listing \\
  --exchange=my-exchange --location=US \\
  --source-dataset=projects/pub-project/datasets/shared_data \\
  --display-name="Shared Analytics Data"
"""

print(analytics_hub)

---
## 6. Cloud DLP and Dynamic Data Masking

In [ ]:
# --- Dynamic Data Masking Simulation ---

customers = pd.DataFrame({
    'customer_id': range(1, 6),
    'name': ['Alice Smith', 'Bob Jones', 'Carol White', 'Dave Brown', 'Eve Davis'],
    'email': ['alice@corp.com', 'bob@mail.com', 'carol@org.net', 'dave@work.io', 'eve@test.com'],
    'ssn': ['123-45-6789', '987-65-4321', '111-22-3333', '444-55-6666', '777-88-9999'],
    'salary': [85000, 92000, 78000, 105000, 88000],
    'region': ['US', 'EU', 'US', 'APAC', 'EU'],
})

def apply_masking(df, role='analyst'):
    """Simulate BigQuery dynamic data masking based on user role."""
    masked = df.copy()
    if role == 'analyst':
        # Analysts see masked PII
        masked['email'] = masked['email'].apply(lambda x: x[0] + '***@' + x.split('@')[1])
        masked['ssn'] = 'XXX-XX-' + masked['ssn'].str[-4:]
        masked['salary'] = None  # Nulled out
    elif role == 'manager':
        # Managers see partial masking
        masked['ssn'] = 'XXX-XX-' + masked['ssn'].str[-4:]
    # Admin sees everything (no masking)
    return masked

print("=== Dynamic Data Masking Simulation ===")
print("\nOriginal data (admin view):")
print(customers.to_string(index=False))

print("\nManager view (SSN partially masked):")
print(apply_masking(customers, 'manager').to_string(index=False))

print("\nAnalyst view (email masked, SSN masked, salary nulled):")
print(apply_masking(customers, 'analyst').to_string(index=False))

print("\nImplementation: Policy tags on columns + IAM roles")
print("  datacatalog.categoryFineGrainedReader -> see actual values")
print("  Without this role -> see masked values")

In [ ]:
# --- Cloud DLP SQL for BigQuery ---

dlp_sql = """
-- De-identify a table using Cloud DLP from BigQuery

-- Option 1: Dynamic masking with policy tags
ALTER TABLE `project.dataset.customers`
ALTER COLUMN ssn SET POLICY TAG
  'projects/proj/locations/us/taxonomies/123/policyTags/ssn_tag';

ALTER TABLE `project.dataset.customers`  
ALTER COLUMN email SET POLICY TAG
  'projects/proj/locations/us/taxonomies/123/policyTags/email_tag';

-- Option 2: Create de-identified copy with DLP API
-- gcloud dlp jobs create \\  
--   --project=my-project \\  
--   --inspect-config='{"infoTypes":[{"name":"EMAIL_ADDRESS"},{"name":"US_SOCIAL_SECURITY_NUMBER"}]}' \\  
--   --deidentify-config='{"recordTransformations":{"fieldTransformations":[...]}}' \\  
--   --storage-config='{"bigQueryOptions":{"tableReference":{...}}}'

-- Key DLP de-identification methods:
--   CryptoReplaceFfxFpe: Format-preserving tokenization
--   CryptoHashConfig: One-way hash
--   CharacterMaskConfig: Replace chars with mask (*)
--   DateShiftConfig: Shift dates by random offset
--   BucketingConfig: Replace values with ranges
--   RedactConfig: Remove the value entirely
"""

print("=== Cloud DLP SQL Patterns ===")
print(dlp_sql)

---
## Summary

| Topic | Key Takeaway |
|-------|-------------|
| **BI Engine** | In-memory cache for sub-second BQ queries. Reserve capacity per project. Preferred tables for priority. |
| **Materialized Views** | Pre-compute aggregations. Auto-rewrite. max_staleness for zero-compute reads. |
| **BigQuery ML** | CREATE MODEL, ML.EVALUATE, ML.PREDICT, ML.GLOBAL_EXPLAIN. All in SQL. |
| **TRANSFORM** | Embed feature engineering in model. Applied automatically at prediction time. |
| **Embeddings** | ML.GENERATE_EMBEDDING + VECTOR_SEARCH for semantic search and RAG. |
| **Analytics Hub** | Zero-copy data sharing. Publisher pays storage, subscriber pays compute. |
| **Dynamic Masking** | Policy tags + IAM roles. Different users see different masking levels on same table. |
| **Cloud DLP** | Inspect (find PII), de-identify (mask/tokenize), risk analysis (k-anonymity). |